# ComputerVisionAIHub · Train a RT-DETR model and publish it

This notebook takes you end-to-end:

1. Pull a labelled dataset from **Roboflow Universe**
2. Train a **RT-DETR** model
3. Export both `.pt` and `.onnx`
4. Push the weights to the **Hugging Face Hub**
5. Auto-print the `models.json` entry to paste into your catalog

> **Before you start:** Runtime → Change runtime type → **T4 GPU** (free).

## 0. Install dependencies

In [1]:
!nvidia-smi

Thu Aug 13 17:01:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q ultralytics huggingface_hub
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q git+https://github.com/roboflow/supervision.git
!pip install -q accelerate
!pip install -q roboflow
!pip install -q torchmetrics
!pip install -q "albumentations>=1.4.5"

Imports

In [3]:
import torch
import requests

import numpy as np
import supervision as sv
import albumentations as A

from PIL import Image
from pprint import pprint
from roboflow import Roboflow
from dataclasses import dataclass, replace
from google.colab import userdata
from torch.utils.data import Dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    TrainingArguments,
    Trainer
)
from torchmetrics.detection.mean_ap import MeanAveragePrecision

## 1. Get the dataset from Roboflow

Find a dataset on https://universe.roboflow.com, click **Download**, choose format
**YOLOv26** (the current Ultralytics export, fully compatible with YOLO26), and copy the snippet.
Paste your details below. Keep your API key private — don't commit it.

In [ ]:

from roboflow import Roboflow
rf = Roboflow(api_key="API_KEY")
project = rf.workspace("ws").project("project_name")
version = project.version(3)
dataset = version.download("yolo26")

In [13]:
DATA_YAML = f"{dataset.location}/data.yaml"
print("data.yaml ->", DATA_YAML)

data.yaml -> /content/OCR-3/data.yaml


## 2. Train


In [ ]:
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")
results = model.train(
    data=DATA_YAML,
    epochs=60,
    batch=16,
    patience=4,
    imgsz=640,
    project="runs",
    name="train"
)

In [ ]:
import numpy as np
# Validation metrics from the best checkpoint
metrics = model.val()
mAP50    = float(metrics.box.map50)
mAP50_95 = float(metrics.box.map)
precision = float(np.mean(metrics.box.p))
recall = float(np.mean(metrics.box.r))
print(f"mAP@50 = {mAP50:.4f} | mAP@50-95 = {mAP50_95:.4f} | precision ={precision:.4f} | recall = {recall:.4f}")

# Export a portable ONNX version (no Ultralytics needed to run it)
onnx_path = model.export(format="onnx")
print("ONNX ->", onnx_path)

import os, glob

BEST_PT   = "/content/runs/detect/runs/train/weights/best.pt"
BEST_ONNX = "/content/runs/detect/runs/train/weights/best.onnx"
#print("pt size (MB):", round(os.path.getsize(BEST_PT)/1e6, 1))

## 4. Publish to the Hugging Face Hub

Create a **Write** token at huggingface.co → Settings → Access Tokens.
We create a public model repo and upload both files.

In [ ]:
from huggingface_hub import login, HfApi, ModelCard

HF_USER  = "USERNAME"          # <- HF username
MODEL_ID = "license-plate-ocr-detector-rt-detr"  # <- unique id for this model
REPO     = f"{HF_USER}/{MODEL_ID}"

login(token="TOKEN")  # paste WRITE token

api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True)
api.upload_file(path_or_fileobj=BEST_PT,   path_in_repo="best.pt",   repo_id=REPO)
api.upload_file(path_or_fileobj=BEST_ONNX, path_in_repo="best.onnx", repo_id=REPO)

# A minimal model card so the HF page isn't empty
card = ModelCard(f"""---
tags: [detection, rt-detr, ultralytics]
license: agpl-3.0
---

# {MODEL_ID}

RT-DETR model trained to perform the OCR on license plate images. Part of the ComputerVisionAIHub catalog.
""")
card.push_to_hub(REPO)
print("Published ->", f"https://huggingface.co/{REPO}")

## 5. Get your `models.json` entry

Fill in the human-readable fields, run the cell, and paste the printed object into
`docs/models.json` (inside the `models` array). That's the only edit the website needs.

In [ ]:
import json

# class names come straight from the trained model
names = list(model.names.values())

entry = {
    "id": MODEL_ID,
    "name": "Raspberry Pi elements detector RT-DETR",            # <- edit
    "summary": "Performs the detection on main elements on Arduino Uno.",     # <- edit
    "base_model": "rtdetr-l",
    "task": "detection",
    "classes": names,
    "dataset": "https:",  # <- edit
    "dataset_license": "CC BY 4.0",                 # <- edit
    "metrics": {"mAP50": round(mAP50, 3), "mAP50_95": round(mAP50_95, 3), "precision": round(precision, 3), "recall": round(recall,3)},
    "hf_repo": REPO,
    "download": f"https://huggingface.co/{REPO}/resolve/main/best.pt",
    "onnx": f"https://huggingface.co/{REPO}/resolve/main/best.onnx",
    "size_mb": round(os.path.getsize(BEST_PT)/1e6, 1),
    "image_size": 640,
}
print(json.dumps(entry, indent=2))

## 6. Try the model

Try the model using images uploaded to the Google Disk.

In [ ]:
prediction = model.predict("PATH", save= True)